# 1. Tokenize FineWeb-Edu into training shards (CPU notebook)

Settings: **Accelerator: None** (uses no GPU quota), **Internet: On**.
Run with **Save Version → Save & Run All (Commit)**. Afterwards, open the
version's **Output** and choose **New Dataset** (name it `fineweb-edu-tokens-32k`).

Downloads FineWeb-Edu `sample/10BT` files 000–003 for training (~8.6 GB) and
file 013 for validation, then writes ~2.1B training tokens (~4.2 GB) plus 10M
validation tokens to `/kaggle/working/tokens`.

In [ ]:
!rm -rf /tmp/repo && git clone --depth 1 https://github.com/V1jey04/llm-from-scratch-124m.git /tmp/repo
!cd /tmp/repo && git log --oneline -1
!pip install -q pyarrow huggingface_hub

In [ ]:
!nproc && free -g && df -h /tmp /kaggle/working | tail -2

In [ ]:
!cd /tmp/repo && python scripts/prepare_data.py \
    --tokenizer tokenizers/fineweb_edu_32k.json --out /kaggle/working/tokens \
    --hf-train-files 000_00000 001_00000 002_00000 003_00000 --hf-val-file 013_00000 \
    --train-tokens 2_100_000_000 --val-tokens 10_000_000 --hf-cache /tmp/hf

In [ ]:
import json
manifest = json.load(open("/kaggle/working/tokens/manifest.json"))
for split, info in manifest["splits"].items():
    print(split, f"{info['tokens']:,} tokens", f"{info['documents']:,} docs", len(info["shards"]), "shards")
print("tokenizer sha256", manifest["tokenizer"]["sha256"][:16])
!du -sh /kaggle/working/tokens && rm -rf /tmp/hf